In [ ]:
"""
Animate AusENDVI NDVI data (every 5 years) for NSW, QLD, and VIC.

Requirements:
    pip install xarray netCDF4 matplotlib cartopy numpy

Usage:
    python ausendvi_animation.py

Notes:
    - Update NC_FILE below to point at your downloaded AusENDVI .nc file.
    - Variable/coordinate names below match the official AusENDVI notebook:
      data var  = "AusENDVI_clim_MCD43A4"
      coords    = "longitude", "latitude", "time"
      QC layer  = "QC" (0 = good obs, 1 = gapfilled, 2 = no-data/ocean)
    - Output is saved as an animated GIF; change ANIM_OUT to .mp4 if you
      have ffmpeg installed and want a video instead.
"""

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
import h5py

# ------------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------------
NC_FILE = "AusENDVI-clim_MCD43A4_gapfilled_1982_2022_0.1.0.nc"
ANIM_OUT = "ndvi_nsw_qld_vic_5yr.gif"
STATES = ["New South Wales", "Queensland", "Victoria"]
YEAR_STEP = 5
MASK_GAPFILLED = False  # set True to blank out pixels that were gap-filled (QC == 1)

NDVI_VAR = "AusENDVI_clim_MCD43A4"
LON, LAT, TIME = "longitude", "latitude", "time"

# Bounding box roughly covering NSW + QLD + VIC (crops before plotting)
LON_MIN, LON_MAX = 138, 154
LAT_MIN, LAT_MAX = -39, -9

# ------------------------------------------------------------------
# 1. LOAD DATA
# ------------------------------------------------------------------
ds = xr.open_dataset(NC_FILE, engine="netcdf4")
da = ds[NDVI_VAR]

if MASK_GAPFILLED:
    good = ds["QC"] != 1
    da = da.where(good)

# ------------------------------------------------------------------
# 2. CROP TO NSW / QLD / VIC BOUNDING BOX
# (latitude runs -10 -> -44, i.e. descending, so slice high->low)
# ------------------------------------------------------------------
da = da.sel({LON: slice(LON_MIN, LON_MAX), LAT: slice(LAT_MAX, LAT_MIN)})

# ------------------------------------------------------------------
# 3. RESAMPLE TO ANNUAL MEAN, THEN PICK EVERY 5 YEARS
# ------------------------------------------------------------------
annual = da.groupby(f"{TIME}.year").mean(dim=TIME)
years_available = annual["year"].values
years_to_plot = years_available[::YEAR_STEP]  # e.g. 1982, 1987, 1992, ...
annual_5yr = annual.sel(year=years_to_plot)

print("Years in animation:", years_to_plot)

# ------------------------------------------------------------------
# 4. STATE BOUNDARIES (outline overlay)
# ------------------------------------------------------------------
shp_path = shpreader.natural_earth(
    resolution="10m", category="cultural", name="admin_1_states_provinces"
)
reader = shpreader.Reader(shp_path)
state_geoms = [
    rec.geometry
    for rec in reader.records()
    if rec.attributes.get("admin") == "Australia"
    and rec.attributes.get("name") in STATES
]

# ------------------------------------------------------------------
# 5. BUILD THE ANIMATION
# ------------------------------------------------------------------
vmin, vmax = float(annual_5yr.min()), float(annual_5yr.max())

fig = plt.figure(figsize=(8, 7))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_geometries(
    state_geoms, ccrs.PlateCarree(),
    facecolor="none", edgecolor="black", linewidth=1.2, zorder=5
)

frame0 = annual_5yr.isel(year=0)
mesh = ax.pcolormesh(
    frame0[LON], frame0[LAT], frame0.values,
    cmap="YlGn", vmin=vmin, vmax=vmax,
    transform=ccrs.PlateCarree()
)
cbar = plt.colorbar(mesh, ax=ax, orientation="vertical", shrink=0.7, pad=0.05)
cbar.set_label("NDVI")
title = ax.set_title(f"NDVI \u2014 {int(years_to_plot[0])}", fontsize=14)


def update(i):
    frame = annual_5yr.isel(year=i)
    mesh.set_array(frame.values.ravel())
    title.set_text(f"NDVI \u2014 {int(years_to_plot[i])}")
    return mesh, title


ani = animation.FuncAnimation(
    fig, update, frames=len(years_to_plot), interval=1000, blit=False
)

ani.save(ANIM_OUT, writer="pillow", fps=1)
print(f"Saved animation to {ANIM_OUT}")

plt.show()

ImportError: No module named 'h5py', backend not available. Please install 'h5py' into your Python environment.

In [14]:
import h5py
